# 🎯 Target Guided Ordinal Encoding

We already know:

| Encoding | When To Use |
|----------|------------|
| Label Encoding | Nominal + tree models |
| Ordinal Encoding | When natural order exists |
| One Hot Encoding | Nominal + linear models |

But what if:

- Categories have **no natural order**
- But you still want to **encode them as numbers**
- And make those numbers **actually meaningful** for the model?

> ## ✅ Solution: Target Guided Ordinal Encoding

---

## 💡 Core Idea

Instead of assigning order **manually** or **randomly**, we use the **target variable** (the column we are trying to predict) to **decide the order**.

**Formula:**

> Calculate the **mean of the target** for each category.
> Rank categories by that mean.
> Assign ordinal numbers based on that rank.

This way the encoding is:
- **Data-driven** — not guesswork
- **Meaningful** — higher number = higher target mean
- **Useful for the model** — captures real relationship

---

## 📌 Basic Example

Suppose dataset:

| City | Salary (Target) |
|------|----------------|
| Lahore | 30000 |
| Karachi | 50000 |
| Lahore | 35000 |
| Islamabad | 70000 |
| Karachi | 45000 |
| Islamabad | 80000 |

### Step 1: Calculate Mean Salary Per City

| City | Mean Salary |
|------|------------|
| Lahore | 32500 |
| Karachi | 47500 |
| Islamabad | 75000 |

### Step 2: Rank By Mean (Ascending)

| City | Mean Salary | Rank |
|------|------------|------|
| Lahore | 32500 | 1 |
| Karachi | 47500 | 2 |
| Islamabad | 75000 | 3 |

### Step 3: Replace City With Rank

| City | Salary | Encoded |
|------|--------|---------|
| Lahore | 30000 | 1 |
| Karachi | 50000 | 2 |
| Lahore | 35000 | 1 |
| Islamabad | 70000 | 3 |
| Karachi | 45000 | 2 |
| Islamabad | 80000 | 3 |

Now the model sees:
```
Islamabad (3) > Karachi (2) > Lahore (1)
```

And this ordering is **backed by real salary data**. ✅

---

## 🔄 Why Not Just Use OHE or Label Encoding Here?

| Method | Problem |
|--------|---------|
| Label Encoding | Assigns random order — Lahore=0, Karachi=1 by alphabet, not by salary |
| One Hot Encoding | Creates 3 new columns, no relationship with salary captured |
| **Target Guided OE** | Creates 1 column, order reflects actual salary relationship ✅ |

---

## 🧪 Dry Run — Step By Step

Suppose dataset:

| Department | Revenue (Target) |
|------------|-----------------|
| HR | 200 |
| Tech | 900 |
| HR | 250 |
| Sales | 600 |
| Tech | 850 |
| Sales | 700 |
| HR | 150 |

### Step 1: Compute Mean Revenue Per Department

| Department | Values | Mean |
|------------|--------|------|
| HR | 200, 250, 150 | 200 |
| Sales | 600, 700 | 650 |
| Tech | 900, 850 | 875 |

### Step 2: Sort By Mean (Ascending)

| Department | Mean Revenue | Rank |
|------------|-------------|------|
| HR | 200 | 1 |
| Sales | 650 | 2 |
| Tech | 875 | 3 |

### Step 3: Map Back To Original Dataset

| Department | Revenue | Encoded |
|------------|---------|---------|
| HR | 200 | 1 |
| Tech | 900 | 3 |
| HR | 250 | 1 |
| Sales | 600 | 2 |
| Tech | 850 | 3 |
| Sales | 700 | 2 |
| HR | 150 | 1 |

---

## 🐍 Python Code

### Method 1: Using pandas — Manual Step By Step

```python
import pandas as pd

This method is same as What I did in 1st code cell, just without ranking,  you can mix both methods like I would use in code cell 2

df = pd.DataFrame({
    'City'   : ['Lahore', 'Karachi', 'Lahore', 'Islamabad', 'Karachi', 'Islamabad'],
    'Salary' : [30000, 50000, 35000, 70000, 45000, 80000]
})

print(df)
```

Output:

| City | Salary |
|------|--------|
| Lahore | 30000 |
| Karachi | 50000 |
| Lahore | 35000 |
| Islamabad | 70000 |
| Karachi | 45000 |
| Islamabad | 80000 |

```python
# Step 1: Compute mean target per category
mean_salary = df.groupby('City')['Salary'].mean()

print(mean_salary)
```

Output:

```
City
Islamabad    75000.0
Karachi      47500.0
Lahore       32500.0
Name: Salary, dtype: float64
```

```python
# Step 2: Rank categories by mean (ascending)
ordinal_map = mean_salary.rank().astype(int)

print(ordinal_map)
```

Output:

```
City
Islamabad    3
Karachi      2
Lahore       1
Name: Salary, dtype: int64
```

```python
# Step 3: Map back to original dataframe
df['City_Encoded'] = df['City'].map(ordinal_map)

print(df)
```

Output:

| City | Salary | City\_Encoded |
|------|--------|--------------|
| Lahore | 30000 | 1 |
| Karachi | 50000 | 2 |
| Lahore | 35000 | 1 |
| Islamabad | 70000 | 3 |
| Karachi | 45000 | 2 |
| Islamabad | 80000 | 3 |

---

### Method 2: Clean Reusable Function

```python
import pandas as pd

def target_guided_ordinal_encoding(df, feature, target):
    """
    Encodes a categorical feature using mean of target variable.

    Parameters:
        df      : pandas DataFrame
        feature : column name to encode (string)
        target  : target column name (string)

    Returns:
        df with new encoded column added
    """

    # Step 1: Compute mean target per category
    mean_target = df.groupby(feature)[target].mean()

    # Step 2: Rank by mean (ascending → lowest mean = rank 1)
    ordinal_map = mean_target.rank(method='dense').astype(int)

    # Step 3: Map to original dataframe
    df[feature + '_Encoded'] = df[feature].map(ordinal_map)

    return df, ordinal_map


# Example usage
df = pd.DataFrame({
    'City'   : ['Lahore', 'Karachi', 'Lahore', 'Islamabad', 'Karachi', 'Islamabad'],
    'Salary' : [30000, 50000, 35000, 70000, 45000, 80000]
})

df, mapping = target_guided_ordinal_encoding(df, 'City', 'Salary')

print(df)
print()
print("Encoding Map:")
print(mapping)
```

Output:

```
        City  Salary  City_Encoded
0     Lahore   30000             1
1    Karachi   50000             2
2     Lahore   35000             1
3  Islamabad   70000             3
4    Karachi   45000             2
5  Islamabad   80000             3

Encoding Map:
City
Islamabad    3
Karachi      2
Lahore       1
Name: Salary, dtype: int64
```

---

### Method 3: Applying To Multiple Columns

```python
import pandas as pd

df = pd.DataFrame({
    'City'       : ['Lahore', 'Karachi', 'Islamabad', 'Lahore', 'Karachi'],
    'Department' : ['HR', 'Tech', 'Sales', 'Tech', 'HR'],
    'Salary'     : [30000, 90000, 60000, 85000, 40000]
})

# Columns to encode
cat_cols = ['City', 'Department']

for col in cat_cols:
    mean_target  = df.groupby(col)['Salary'].mean()
    ordinal_map  = mean_target.rank(method='dense').astype(int)
    df[col + '_Encoded'] = df[col].map(ordinal_map)

print(df)
```

Output:

| City | Department | Salary | City\_Encoded | Department\_Encoded |
|------|------------|--------|--------------|-------------------|
| Lahore | HR | 30000 | 1 | 1 |
| Karachi | Tech | 90000 | 2 | 3 |
| Islamabad | Sales | 60000 | 3 | 2 |
| Lahore | Tech | 85000 | 1 | 3 |
| Karachi | HR | 40000 | 2 | 1 |

---

### Method 4: Train / Test Split Safe Version

> ⚠️ Very Important:
> Always learn the encoding **only on training data**.
> Then apply the same mapping to test data.
> Never fit on test data — it causes **data leakage**.

```python
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.DataFrame({
    'City'   : ['Lahore', 'Karachi', 'Lahore', 'Islamabad',
                'Karachi', 'Islamabad', 'Lahore', 'Karachi'],
    'Salary' : [30000, 50000, 35000, 70000,
                45000, 80000, 32000, 48000]
})

# Split into train and test
train, test = train_test_split(df, test_size=0.25, random_state=42)

# Step 1: Learn mapping FROM TRAINING DATA ONLY
mean_target = train.groupby('City')['Salary'].mean()
ordinal_map = mean_target.rank(method='dense').astype(int)

print("Encoding map (learned from train):")
print(ordinal_map)

# Step 2: Apply same mapping to both train and test
train['City_Encoded'] = train['City'].map(ordinal_map)
test['City_Encoded']  = test['City'].map(ordinal_map)

print("\nTrain:")
print(train)

print("\nTest:")
print(test)
```

---

## ⚠️ Data Leakage Warning

```
❌ WRONG: Fit encoding on entire dataset before splitting
✅ CORRECT: Split first → Fit encoding on train only → Apply to test
```

| Mistake | Consequence |
|---------|------------|
| Encoding on full data | Test data leaks into training — overly optimistic results |
| Encoding on train only | Safe — real world performance is honest |

---

## ✅ When To Use Target Guided Ordinal Encoding

| Situation | Use It? |
|-----------|---------|
| Nominal categories with high cardinality | ✅ Yes |
| No natural order exists | ✅ Yes |
| Want to capture target relationship | ✅ Yes |
| Regression or binary classification tasks | ✅ Yes |
| Categories have natural order already | ❌ Use Ordinal Encoding |
| Very small dataset (unstable means) | ❌ Risky |

---

## 👍 Advantages

| # | Advantage |
|---|-----------|
| 1 | **Data-driven** — order based on real target relationship |
| 2 | **No extra columns** — memory efficient like label encoding |
| 3 | **Captures relationship** between category and target |
| 4 | **Works well** with both tree and linear models |
| 5 | **Handles high cardinality** better than OHE |

---

## 👎 Disadvantages

| # | Disadvantage |
|---|-------------|
| 1 | **Risk of data leakage** if not done carefully on train only |
| 2 | **Unstable with small datasets** — means can be noisy |
| 3 | **Overfitting risk** — encoding is too specific to training data |
| 4 | **Only uses mean** — ignores variance inside categories |

---

## 🧠 Key Rule To Remember

> **Target Guided Ordinal Encoding = Let the data decide the order**

```
Manual order exists   → Ordinal Encoding
No order, few cats    → One Hot Encoding
No order, many cats   → Target Guided Ordinal Encoding ✅
```

| Method | Order Source | Extra Columns | Captures Target? |
|--------|-------------|---------------|-----------------|
| Label Encoding | Alphabetical (random) | ❌ No | ❌ No |
| Ordinal Encoding | You define manually | ❌ No | ❌ No |
| One Hot Encoding | No order | ✅ Yes (many) | ❌ No |
| **Target Guided OE** | **Target mean** | ❌ **No** | ✅ **Yes** |

In [1]:
import numpy as np
import pandas as pd


In [8]:
import pandas as pd

# =========================================================
# 🧠 STEP 1: CREATE DATASET (CITY vs PRICE)
# =========================================================
# We have:
# - City (categorical feature)
# - Price (target influence)
# ---------------------------------------------------------

df = pd.DataFrame({
    "City": ["Lahore", "Karachi", "Lahore", "Multan", "Karachi", "Multan"],
    "Price": [100, 200, 150, 80, 220, 90]
})

print("ORIGINAL DATA:")
print(df)


# =========================================================
# 🧠 STEP 2: COMPUTE MEAN PRICE FOR EACH CITY
# =========================================================
# IDEA:
# We want to see:
# 👉 Which city has higher average price?
# ---------------------------------------------------------

city_mean = df.groupby("City")["Price"].mean()

print("\nMean Price per City (Series):")
print(city_mean)


# =========================================================
# 🧠 STEP 3: CONVERT SERIES → DICTIONARY
# =========================================================
# WHY?
# - Dictionary is faster for mapping
# - Easier lookup: City → Value
# ---------------------------------------------------------

city_mean = city_mean.to_dict()

print("\nCITY → MEAN PRICE MAPPING:")
print(city_mean)


# =========================================================
# 🧠 STEP 4: MAP VALUES BACK TO DATAFRAME
# =========================================================
# We replace City with its MEAN PRICE value
# This is TARGET GUIDED ENCODING
# ---------------------------------------------------------

df["mean_price"] = df["City"].map(city_mean)


# =========================================================
# 🧠 STEP 5: FINAL OUTPUT
# =========================================================
# Now each city is replaced by its statistical signal
# (mean price)
# ---------------------------------------------------------

print("\nFINAL DATA WITH TARGET ENCODING:")
print(df)

ORIGINAL DATA:
      City  Price
0   Lahore    100
1  Karachi    200
2   Lahore    150
3   Multan     80
4  Karachi    220
5   Multan     90

Mean Price per City (Series):
City
Karachi    210.0
Lahore     125.0
Multan      85.0
Name: Price, dtype: float64

CITY → MEAN PRICE MAPPING:
{'Karachi': 210.0, 'Lahore': 125.0, 'Multan': 85.0}

FINAL DATA WITH TARGET ENCODING:
      City  Price  mean_price
0   Lahore    100       125.0
1  Karachi    200       210.0
2   Lahore    150       125.0
3   Multan     80        85.0
4  Karachi    220       210.0
5   Multan     90        85.0


---

# Giving Rank to Avg or mean col 
     
     lower value -> lower rank i.e 1 
     higher value -> higher rank i.e 2

In [ ]:
import pandas as pd

# =========================================================
# 🧠 STEP 1: CREATE DATASET
# =========================================================

df = pd.DataFrame({
    'City'   : ['Lahore', 'Karachi', 'Lahore', 'Islamabad', 'Karachi', 'Islamabad'],
    'Salary' : [30000, 50000, 35000, 70000, 45000, 80000]
})

print("ORIGINAL DATA:")
print(df)


# =========================================================
# 🧠 STEP 2: COMPUTE MEAN SALARY PER CITY
# =========================================================

city_mean = df.groupby("City")["Salary"].mean()

print("\nCITY MEAN:")
print(city_mean)


# =========================================================
# 🧠 STEP 3: CREATE CITY MEAN COLUMN (MAP BACK)
# =========================================================

df["City_Mean"] = df["City"].map(city_mean)

print("\nDATA WITH CITY MEAN:")
print(df)


# =========================================================
# 🧠 STEP 4: CREATE PROPER CITY RANK (FIXED LOGIC)
# =========================================================
# IMPORTANT:
# We rank ONLY unique city means, not full column
# =========================================================

city_rank = city_mean.rank(method="dense").astype(int)
# dense:
# ensures ranks are continuous (no gaps)
# example: 1,2,3 NOT 1,3,5

# default rank method is avg which will give 1,3,5 ( MOdel will think that there is very Huge differnce b/w Values)

print("\nCITY RANK MAPPING:")
print(city_rank)


# =========================================================
# 🧠 STEP 5: MAP RANK BACK TO DATAFRAME
# =========================================================

df["City_Rank"] = df["City"].map(city_rank)

print("\nFINAL DATA WITH MEAN + RANK:")
print(df)

ORIGINAL DATA:
        City  Salary
0     Lahore   30000
1    Karachi   50000
2     Lahore   35000
3  Islamabad   70000
4    Karachi   45000
5  Islamabad   80000

CITY MEAN:
City
Islamabad    75000.0
Karachi      47500.0
Lahore       32500.0
Name: Salary, dtype: float64

DATA WITH CITY MEAN:
        City  Salary  City_Mean
0     Lahore   30000    32500.0
1    Karachi   50000    47500.0
2     Lahore   35000    32500.0
3  Islamabad   70000    75000.0
4    Karachi   45000    47500.0
5  Islamabad   80000    75000.0

CITY RANK MAPPING:
City
Islamabad    3
Karachi      2
Lahore       1
Name: Salary, dtype: int64

FINAL DATA WITH MEAN + RANK:
        City  Salary  City_Mean  City_Rank
0     Lahore   30000    32500.0          1
1    Karachi   50000    47500.0          2
2     Lahore   35000    32500.0          1
3  Islamabad   70000    75000.0          3
4    Karachi   45000    47500.0          2
5  Islamabad   80000    75000.0          3


In [13]:
import pandas as pd

# =========================================================
# 🧠 STEP 1: CREATE DATASET
# =========================================================
# We have:
# - Department  → categorical feature
# - Bonus       → target / numeric signal
#
# Goal:
# Convert Department into numbers based on:
# 👉 average bonus performance
# =========================================================

df = pd.DataFrame({
    "Department": [
        "HR",
        "IT",
        "Finance",
        "IT",
        "HR",
        "Finance",
        "Marketing",
        "Marketing"
    ],

    "Bonus": [
        20000,
        50000,
        70000,
        55000,
        25000,
        80000,
        40000,
        45000
    ]
})

print("ORIGINAL DATA:")
print(df)


# =========================================================
# 🧠 STEP 2: FIND MEAN BONUS PER DEPARTMENT
# =========================================================
# groupby("Department")
# → groups rows department-wise
#
# ["Bonus"].mean()
# → computes average bonus for each department
#
# RESULT:
# HR         → average bonus
# IT         → average bonus
# Finance    → average bonus
# Marketing  → average bonus
# =========================================================

dept_mean = df.groupby("Department")["Bonus"].mean()

print("\nMEAN BONUS PER DEPARTMENT:")
print(dept_mean)


# =========================================================
# 🧠 STEP 3: MAP MEAN BACK TO ORIGINAL DATAFRAME
# =========================================================
# .map()
# → works like dictionary lookup
#
# Example:
# HR → 22500
# IT → 52500
#
# Every department gets replaced by its mean bonus
# =========================================================

df["Dept_Mean"] = df["Department"].map(dept_mean)

print("\nDATA WITH MEAN COLUMN:")
print(df)


# =========================================================
# 🧠 STEP 4: CREATE RANKING
# =========================================================
# WHY RANK?
#
# Mean values can be very large:
# 22500, 52500, 75000 ...
#
# Models sometimes learn better from:
# simple ordered levels:
#
# 1,2,3,4
#
# So now we convert:
#
# lowest mean  → 1
# higher mean  → 2
# highest mean → 4
# =========================================================


# =========================================================
# IMPORTANT:
# We rank ONLY UNIQUE department means
#
# NOT full dataframe column
#
# Because:
# Each department should get ONE fixed rank
# =========================================================

dept_rank = dept_mean.rank(method="dense").astype(int)

# method="dense"
#
# ensures:
# 1,2,3,4
#
# NOT:
# 1,3,5,7
#
# dense ranking means:
# no gaps in rank numbers


# default rank method = average
#
# average ranking can create gaps
# because duplicate values share average positions
#
# ML models may incorrectly think:
# rank 1 and rank 5 have HUGE distance
#
# dense keeps ranking clean and continuous

print("\nDEPARTMENT RANK MAPPING:")
print(dept_rank)


# =========================================================
# 🧠 STEP 5: MAP RANK BACK TO DATAFRAME
# =========================================================
# Replace each department with its rank
#
# Example:
# HR        → 1
# Marketing → 2
# IT        → 3
# Finance   → 4
# =========================================================

df["Dept_Rank"] = df["Department"].map(dept_rank)

print("\nFINAL DATA:")
print(df)

ORIGINAL DATA:
  Department  Bonus
0         HR  20000
1         IT  50000
2    Finance  70000
3         IT  55000
4         HR  25000
5    Finance  80000
6  Marketing  40000
7  Marketing  45000

MEAN BONUS PER DEPARTMENT:
Department
Finance      75000.0
HR           22500.0
IT           52500.0
Marketing    42500.0
Name: Bonus, dtype: float64

DATA WITH MEAN COLUMN:
  Department  Bonus  Dept_Mean
0         HR  20000    22500.0
1         IT  50000    52500.0
2    Finance  70000    75000.0
3         IT  55000    52500.0
4         HR  25000    22500.0
5    Finance  80000    75000.0
6  Marketing  40000    42500.0
7  Marketing  45000    42500.0

DEPARTMENT RANK MAPPING:
Department
Finance      4
HR           1
IT           3
Marketing    2
Name: Bonus, dtype: int64

FINAL DATA:
  Department  Bonus  Dept_Mean  Dept_Rank
0         HR  20000    22500.0          1
1         IT  50000    52500.0          3
2    Finance  70000    75000.0          4
3         IT  55000    52500.0          3
4    

---

# ML PipeLine 

In [14]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder
)

# =========================================================
# 🧠 STEP 1: CREATE DATASET
# =========================================================
# TYPES OF FEATURES:
#
# City              → Nominal
# Gender            → Nominal
#
# Education         → Ordinal
# Experience_Level  → Ordinal
#
# Department        → Target Guided Encoding
#
# Selected          → Target (y)
# =========================================================

df = pd.DataFrame({

    "City": ["Lahore", "Karachi", "Islamabad", "Lahore", "Karachi", "Multan"],

    "Gender": ["Male", "Female", "Male", "Female", "Male", "Female"],

    "Education": ["Bachelor", "Master", "PhD", "HighSchool", "Bachelor", "Master"],

    "Experience_Level": ["Junior", "Senior", "Expert", "Junior", "Senior", "Expert"],

    "Department": ["IT", "HR", "Finance", "IT", "Finance", "HR"],

    "Salary": [50000, 30000, 90000, 55000, 85000, 35000],

    "Selected": ["Yes", "No", "Yes", "No", "Yes", "No"]
})

print("ORIGINAL DATA:")
print(df)


# =========================================================
# 🧠 STEP 2: TRAIN / TEST SPLIT
# =========================================================

train = df.iloc[:4]

test = pd.DataFrame({

    "City": ["Lahore", "NewCity"],

    "Gender": ["Male", "Female"],

    "Education": ["Bachelor", "Diploma"],

    "Experience_Level": ["Junior", "UltraExpert"],

    "Department": ["Finance", "Marketing"],

    "Salary": [60000, 40000],

    "Selected": ["Yes", "No"]
})

print("\nTRAIN DATA:")
print(train)

print("\nTEST DATA:")
print(test)


# =========================================================
# 🧠 STEP 3: SPLIT FEATURES (X) AND TARGET (y)
# =========================================================

X_train = train.drop("Selected", axis=1)
X_test = test.drop("Selected", axis=1)

y_train = train["Selected"]
y_test = test["Selected"]


# =========================================================
# 🧠 STEP 4: LABEL ENCODING (TARGET)
# =========================================================

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)

y_test_encoded = label_encoder.transform(y_test)

print("\nENCODED TARGET:")
print(y_train_encoded)


# =========================================================
# 🧠 STEP 5: ONE HOT ENCODING
# =========================================================
# Used for:
# City + Gender
#
# Because:
# NO ORDER
# =========================================================

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_onehot = onehot_encoder.fit_transform(
    X_train[["City", "Gender"]]
)

X_test_onehot = onehot_encoder.transform(
    X_test[["City", "Gender"]]
)

print("\nONE HOT TRAIN:")
print(X_train_onehot)


# =========================================================
# 🧠 STEP 6: ORDINAL ENCODING
# =========================================================
# Used for:
# Education + Experience_Level
#
# Because:
# REAL ORDER EXISTS
# =========================================================

education_order = [
    "HighSchool",
    "Bachelor",
    "Master",
    "PhD"
]

experience_order = [
    "Junior",
    "Senior",
    "Expert"
]

ordinal_encoder = OrdinalEncoder(

    categories=[
        education_order,
        experience_order
    ],

    handle_unknown="use_encoded_value",

    unknown_value=-1
)

X_train_ordinal = ordinal_encoder.fit_transform(

    X_train[[
        "Education",
        "Experience_Level"
    ]]
)

X_test_ordinal = ordinal_encoder.transform(

    X_test[[
        "Education",
        "Experience_Level"
    ]]
)

print("\nORDINAL TRAIN:")
print(X_train_ordinal)


# =========================================================
# 🧠 STEP 7: TARGET GUIDED ENCODING
# =========================================================
# Used for:
# Department
#
# Logic:
# Department → Mean Salary → Rank
# =========================================================

dept_mean = train.groupby("Department")["Salary"].mean()

print("\nDEPARTMENT MEAN:")
print(dept_mean)

dept_rank = dept_mean.rank(method="dense").astype(int)

print("\nDEPARTMENT RANK:")
print(dept_rank)

X_train_target = X_train["Department"].map(dept_rank)

X_test_target = X_test["Department"].map(
    dept_rank
).fillna(-1)

print("\nTARGET GUIDED TRAIN:")
print(X_train_target)


# =========================================================
# 🧠 STEP 8: COMBINE ALL FEATURES
# =========================================================

X_train_final = np.hstack([

    X_train_onehot,

    X_train_ordinal,

    X_train_target.values.reshape(-1, 1)
])

X_test_final = np.hstack([

    X_test_onehot,

    X_test_ordinal,

    X_test_target.values.reshape(-1, 1)
])


# =========================================================
# 🧠 STEP 9: CONVERT FINAL ARRAYS → DATAFRAMES
# =========================================================
# Horizontal readable output
# =========================================================

onehot_cols = onehot_encoder.get_feature_names_out([
    "City",
    "Gender"
])

final_cols = list(onehot_cols) + [

    "Education_Encoded",

    "Experience_Encoded",

    "Department_TargetRank"
]

train_final_df = pd.DataFrame(
    X_train_final,
    columns=final_cols
)

test_final_df = pd.DataFrame(
    X_test_final,
    columns=final_cols
)


# =========================================================
# 🧠 STEP 10: FINAL OUTPUT
# =========================================================

print("\nFINAL TRAIN DATAFRAME:")
print(train_final_df)

print("\nFINAL TEST DATAFRAME:")
print(test_final_df)

ORIGINAL DATA:
        City  Gender   Education Experience_Level Department  Salary Selected
0     Lahore    Male    Bachelor           Junior         IT   50000      Yes
1    Karachi  Female      Master           Senior         HR   30000       No
2  Islamabad    Male         PhD           Expert    Finance   90000      Yes
3     Lahore  Female  HighSchool           Junior         IT   55000       No
4    Karachi    Male    Bachelor           Senior    Finance   85000      Yes
5     Multan  Female      Master           Expert         HR   35000       No

TRAIN DATA:
        City  Gender   Education Experience_Level Department  Salary Selected
0     Lahore    Male    Bachelor           Junior         IT   50000      Yes
1    Karachi  Female      Master           Senior         HR   30000       No
2  Islamabad    Male         PhD           Expert    Finance   90000      Yes
3     Lahore  Female  HighSchool           Junior         IT   55000       No

TEST DATA:
      City  Gender Educa